# Digital Touchpoints Harmonization

Harmonizes the raw Digital Touchpoint export (`bci_rbs_dev.raw.me_bdo_dto_261116`) into `bci_rbs_dev.harmonized.me_bdo_digital_touchpoint` based on the `DigitalTouchpoints` aspect model. Organizational office and planning object IDs are added by joining the touchpoint owner on the organization department.

In [0]:
%pip install esmf-aspect-model-loader==2.3.0 pydeequ==1.5.0 pydantic==2.13.4 oauthlib==3.3.1 annotated-types==0.7.0 jinja2==3.1.2

In [0]:
%run /Workspace/RBS/.bundle/rbs_python_sdk_0.9.13/files/rbs_python_sdk/set_python_path_for_bundles

In [0]:
import os

# Source (raw layer) and target (harmonized layer)
SOURCE_TABLE = "bci_rbs_dev.raw.me_bdo_dto_261116"
ORGANIZATION_TABLE = "bci_rbs_dev.raw.me_bdo_organization_all"  # Department -> ManagerOrgID / ManagerPlanningObject
TARGET_CATALOG = "bci_rbs_dev"
TARGET_SCHEMA = "harmonized"
TARGET_TABLE = "me_bdo_digital_touchpoint"
TARGET_FQN = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}"

# Aspect model: aspect file + shared files (organization properties live in the organization model)
MODELS_ROOT = "/Workspace/Users/nec9bue@bosch.com/ME_BDO_Reporting/Models"
DTP_MODEL_DIR = f"{MODELS_ROOT}/com.bosch.mobility.mobilityelectronics.businessdigitalorganization.digitaltouchpoint/1.0.0"
ORG_MODEL_DIR = f"{MODELS_ROOT}/com.bosch.mobility.mobilityelectronics.businessdigitalorganization.organization/1.0.0"
TTL_FILES = [
    os.path.join(DTP_MODEL_DIR, "DigitalTouchpoints.ttl"),
    os.path.join(DTP_MODEL_DIR, "DigitalTouchpoint_shared.ttl"),
    os.path.join(DTP_MODEL_DIR, "DigitalTouchpointStatus_shared.ttl"),
    os.path.join(ORG_MODEL_DIR, "OrganizationalOffice_shared.ttl"),
    os.path.join(ORG_MODEL_DIR, "PlanningObject_shared.ttl"),
]

# Raw column -> aspect property column (snake_case, as generated by SammCli)
COLUMN_MAPPING = {
    "ID":               "digital_touchpoint_id",
    "Touchpoint":       "digital_touchpoint_name",
    "Status":           "digital_touchpoint_status",
    "touchpoint_owner": "digital_touchpoint_owner",  # also join key to ORGANIZATION_TABLE.Department
}

# Business key: must be present and numeric (filters the report footer rows), one record per key
REQUIRED_COLUMNS = ["digital_touchpoint_id"]
KEY_PATTERN = r"^\d+$"

# Model properties without a source column (filled with NULL until a source is available)
# organizational_office_id / _name / planning_object_id are derived from ORGANIZATION_TABLE in cell 07
NULLABLE_OVERRIDES = ["lifecycle_phase", "factsheet_id", "factsheet_type", "number_of_open_requirements"]

# Model-required columns that may stay NULL (no source, or owner not found in ORGANIZATION_TABLE)
RELAX_NOT_NULL = NULLABLE_OVERRIDES + ["organizational_office_id", "organizational_office_name", "planning_object_id"]

print(f"Source: {SOURCE_TABLE}\nTarget: {TARGET_FQN}")

In [0]:
from common.aspect_model_loader import AspectModelLoader

# Concatenate aspect + shared TTL so all property definitions resolve in one graph
ttl_content = ""
for path in TTL_FILES:
    assert os.path.exists(path), f"TTL file not found: {path}"
    with open(path, "r") as f:
        ttl_content += f.read() + "\n"

# The factsheet (fact:) and common (bdo-comm:) models are not in the workspace yet. Unresolved OPTIONAL
# properties and dangling samm:extends references are dropped in memory (loader and SammCli cannot
# resolve them); unresolved required properties fail. Add the missing TTL files to TTL_FILES to include them.
import rdflib
from rdflib.collection import Collection
SAMM = rdflib.Namespace("urn:samm:org.eclipse.esmf.samm:meta-model:2.2.0#")
g = rdflib.Graph().parse(data=ttl_content, format="turtle")
aspect_node = next(g.subjects(rdflib.RDF.type, SAMM.Aspect))
props = Collection(g, g.value(aspect_node, SAMM.properties))
keep, dropped = [], []
for item in list(props):
    prop = g.value(item, SAMM.property) or item
    if (prop, rdflib.RDF.type, SAMM.Property) in g:
        keep.append(item)
    else:
        assert g.value(item, SAMM.optional) is not None and g.value(item, SAMM.optional).toPython(), \
            f"Required property without definition: {prop}"
        dropped.append(str(prop))
        g.remove((item, None, None))  # orphan [ samm:property ...; samm:optional true ] node
dangling = [t for t in g.triples((None, SAMM.extends, None)) if (t[2], rdflib.RDF.type, None) not in g]
for t in dangling:
    g.remove(t)
if dropped or dangling:
    props.clear()
    for item in keep:
        props.append(item)
    ttl_content = g.serialize(format="turtle")
    print(f"Dropped unresolved optional properties: {dropped}")
    print(f"Dropped dangling extends:               {[str(t[2]) for t in dangling]}")

loader = AspectModelLoader()
samm_graph = loader.get_aspect_model(model_data_string=ttl_content, source="data_string")
samm_graph.load_aspect_model()

aspect = samm_graph.aspect
print(f"Loaded aspect: {aspect.name}")
print(f"URN:           {aspect.urn}")
print(f"Properties:    {[p.name for p in aspect.properties]}")

In [0]:
import tempfile, subprocess, sys, re

# SammCli.to_sql() generates CREATE TABLE (snake_case columns, types, comments, URN property).
# Run in a subprocess to capture its output reliably in notebooks.
with tempfile.NamedTemporaryFile(mode="w", suffix=".ttl", delete=False) as f:
    f.write(ttl_content)
    _ttl_tmp = f.name
try:
    result = subprocess.run(
        [sys.executable, "-c",
         f"from esmf_aspect_meta_model_python.samm_cli import SammCli; SammCli().to_sql('{_ttl_tmp}')"],
        capture_output=True, text=True,
    )
    assert result.returncode == 0, f"SammCli error:\n{result.stderr}"
    generated_sql = result.stdout.strip().rstrip(";")
finally:
    os.unlink(_ttl_tmp)

# Point the DDL at the target table
create_table_sql = re.sub(r"CREATE TABLE IF NOT EXISTS\s+\S+",
                          f"CREATE TABLE IF NOT EXISTS {TARGET_FQN}", generated_sql, count=1)

# Relax NOT NULL for model properties without a (guaranteed) source value
for col in RELAX_NOT_NULL:
    create_table_sql = re.sub(rf"(\b{col}\s+\w+)\s+NOT NULL", r"\1", create_table_sql)

# Add technical columns (not part of the aspect model) before the closing column bracket
create_table_sql = re.sub(
    r"\n\)\s*\nCOMMENT",
    ",\n  _load_date TIMESTAMP NOT NULL COMMENT 'Load timestamp of the harmonization run.',"
    "\n  _record_source STRING NOT NULL COMMENT 'Source table of the record.'\n)\nCOMMENT",
    create_table_sql, count=1,
)

print(create_table_sql)

In [0]:
from pyspark.sql import functions as F, Window

raw_df = spark.read.table(SOURCE_TABLE)

# Rename to aspect property columns; blank strings -> NULL
mapped_df = raw_df.select([
    F.nullif(F.trim(F.col(src).cast("string")), F.lit("")).alias(tgt) for src, tgt in COLUMN_MAPPING.items()
])
# Model properties without a source column
for col in NULLABLE_OVERRIDES:
    mapped_df = mapped_df.withColumn(col, F.lit(None).cast("string"))

# Organizational office (id + name) + planning object per touchpoint owner: owner = organization Department
# (office name = Department, same mapping as Master_Data_Harmonized_Organization)
# (one office per department; active offices preferred; left join keeps owners without a match)
org_df = (
    spark.read.table(ORGANIZATION_TABLE)
    .select(
        F.upper(F.trim("Department")).alias("_owner_key"),
        F.nullif(F.trim("ManagerOrgID"), F.lit("")).alias("organizational_office_id"),
        F.nullif(F.trim("Department"), F.lit("")).alias("organizational_office_name"),
        F.nullif(F.trim("ManagerPlanningObject"), F.lit("")).alias("planning_object_id"),
        F.col("OrgDeleted"),
    )
    .filter(F.col("_owner_key").isNotNull() & F.col("organizational_office_id").isNotNull())
    .withColumn("_rn", F.row_number().over(
        Window.partitionBy("_owner_key").orderBy(F.col("OrgDeleted") == "True", "organizational_office_id")))
    .filter("_rn = 1")
    .drop("_rn", "OrgDeleted")
)
mapped_df = (
    mapped_df.withColumn("_owner_key", F.upper(F.col("digital_touchpoint_owner")))
    .join(org_df, "_owner_key", "left")
    .drop("_owner_key")
)

# Required columns (business key, numeric); all other aspect properties are optional
is_valid = F.lit(True)
for c in REQUIRED_COLUMNS:
    is_valid = is_valid & F.col(c).isNotNull() & F.col(c).rlike(KEY_PATTERN)

# Anomalies: rows without a valid business key, e.g. report footer (kept for review, not written)
anomalies_df = mapped_df.filter(~is_valid).distinct()

# Harmonized: valid rows, one record per business key
harmonized_df = (
    mapped_df.filter(is_valid)
    .dropDuplicates(REQUIRED_COLUMNS)
    .withColumn("_load_date", F.current_timestamp())
    .withColumn("_record_source", F.lit(SOURCE_TABLE))
)

unmatched_df = harmonized_df.filter(F.col("organizational_office_id").isNull())
print(f"Raw rows:          {raw_df.count()}")
print(f"Anomaly rows:      {anomalies_df.count()}")
print(f"Harmonized rows:   {harmonized_df.count()}")
print(f"Owner not matched: {unmatched_df.count()}")
display(anomalies_df)
display(harmonized_df.limit(10))

In [0]:
# Create table from the model-derived DDL (keeps comments, URN property and existing tags)
spark.sql(create_table_sql)

# Schema evolution: add model columns missing in an existing table (nullable, at their model position)
existing_cols = set(spark.table(TARGET_FQN).columns)
prev_col = None
for col_name, col_def in re.findall(r"^\s{2}(\w+)\s+(.+?),?$", create_table_sql, flags=re.M):
    if col_name not in existing_cols:
        col_def = col_def.replace(" NOT NULL", "")
        position = f" AFTER {prev_col}" if prev_col else " FIRST"
        spark.sql(f"ALTER TABLE {TARGET_FQN} ADD COLUMN {col_name} {col_def}{position}")
        print(f"Added column: {col_name}")
    prev_col = col_name

# Overwrite data; columns ordered like the target table (insertInto is positional)
target_cols = spark.table(TARGET_FQN).columns
harmonized_df.select(target_cols).write.insertInto(TARGET_FQN, overwrite=True)

print(f"Written {spark.table(TARGET_FQN).count()} rows to {TARGET_FQN}")
display(spark.sql(f"DESCRIBE TABLE {TARGET_FQN}"))
display(spark.table(TARGET_FQN).limit(5))

In [0]:
from common.aspect_model_client import AspectModelClient
from databricks_client.table_metadata_manager import TableMetadataManager

DRY_RUN = False  # True = only log the generated SQL, do not execute

# Map table columns to aspect model properties
client = AspectModelClient(samm_model=samm_graph, data_frame=spark.table(TARGET_FQN))

manager = TableMetadataManager(
    spark=spark,
    catalog=TARGET_CATALOG,
    schema=TARGET_SCHEMA,
    table_name=TARGET_TABLE,
    log_sql_statements=True,  # show generated COMMENT / SET TAGS statements
)

# Table comment + aspect URN tag, column comments + property URN tags
success, errors = manager.apply_aspect_model_metadata(client, dry_run=DRY_RUN)
print(f"Success: {success}")
for err in errors:
    print(f"[{err.operation}] {err.error}")

# Verify applied tags
display(spark.sql(f"""
    SELECT 'TABLE' AS scope, NULL AS column_name, tag_name, tag_value
    FROM {TARGET_CATALOG}.information_schema.table_tags
    WHERE schema_name = '{TARGET_SCHEMA}' AND table_name = '{TARGET_TABLE}'
    UNION ALL
    SELECT 'COLUMN', column_name, tag_name, tag_value
    FROM {TARGET_CATALOG}.information_schema.column_tags
    WHERE schema_name = '{TARGET_SCHEMA}' AND table_name = '{TARGET_TABLE}'
    ORDER BY scope DESC, column_name
"""))